# Image Inpainting — Restoring Damaged Areas

This notebook walks through the project end to end:

1. Load and preprocess an image
2. Simulate damage (a square region zeroed out)
3. Inspect the encoder-decoder architecture
4. Restore the damaged image with the trained model
5. Measure quality with PSNR

All reusable logic lives in the `inpainting` package under `src/`.

In [ ]:
import sys, pathlib
# Make the src/ package importable when running the notebook in-place.
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from inpainting.data import load_image
from inpainting.masking import add_square_mask
from inpainting.metrics import psnr
from inpainting.visualize import show_comparison

## 1. Load and damage an image

In [ ]:
original = load_image('../assets/sample.jpeg')
damaged, top_left = add_square_mask(original, rng=np.random.default_rng(7))
print('image shape:', original.shape, '| mask top-left:', top_left)

fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
for a, im, t in zip(ax, [original, damaged], ['Original', 'Damaged']):
    a.imshow(im); a.set_title(t); a.axis('off')
plt.show()

## 2. The model

A compact convolutional autoencoder: an encoder compresses the damaged image,
a bottleneck captures context, and a decoder reconstructs the full image.

In [ ]:
from inpainting.model import build_inpainting_model
model = build_inpainting_model()
model.summary()

## 3. Restore with the trained model

Loads `models/inpainting_model.h5` (shipped with the repo) and restores the
damaged image above.

In [ ]:
from inpainting.predict import load_inpainting_model, restore

model = load_inpainting_model('../models/inpainting_model.h5')
restored = restore(model, damaged)

show_comparison(original, damaged, restored)
plt.show()

print('PSNR damaged vs original :', round(psnr(original, damaged), 2), 'dB')
print('PSNR restored vs original:', round(psnr(original, restored), 2), 'dB')

## Next steps

- Train on your own dataset: `python -m inpainting.train --dataset path/to/images`
- Try irregular or larger masks by editing `inpainting/masking.py`
- Add perceptual / SSIM loss for sharper reconstructions